[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Before editing:** choose **File → Save a copy in Drive**.

[Course home](../README.md) · [Learning path](../LEARNING_PATH.md) · [Course overview](../COURSE_OVERVIEW.md) · [Research assignments](../RESEARCH_ASSIGNMENTS.md) · [Setup](../SETUP.md) · [Course links](../COURSE_LINKS.md)

The badge opens Colab. Choose **File → Open notebook → Upload** and upload this `.ipynb` file from your downloaded package. In Colab, keep the downloaded setup guide and course home open separately; relative document links are for GitHub or local viewing. Opening a notebook does not copy its support files. In Colab, run the setup cell below and upload the Python Basics ZIP or your weekly ZIP from the course home. Locally, open this notebook from the extracted `learning_materials/` directory. Keep your own saved copy of assessed work.


<a id="lookup-index"></a>
# Python Basics · Quick lookup

Use this as a 早見帳: find a syntax pattern, inspect one example, then return to your weekly notebook. All required explanations, exercises, project tasks and completion checks are in Week_XX.ipynb. This is not a second homework stream.

- [00 · Execution, comments and indentation](#b00)
- [01 · Variables, types and conversion](#b01)
- [02 · Arithmetic and precedence](#b02)
- [03 · Strings and formatted output](#b03)
- [04 · Comparisons and decisions](#b04)
- [05 · Lists, dictionaries, tuples and sets](#b05)
- [06 · Loops and aggregation](#b06)
- [07 · Functions, arguments and return values](#b07)
- [08 · Reading errors and debugging](#b08)
- [09 · Raising and handling exceptions](#b09)
- [10 · Normal, boundary, invalid and regression tests](#b10)
- [11 · Imports, files and CSV](#b11)
- [12 · JSON, missing values and validation](#b12)
- [13 · SQLite, transactions and state after failure](#b13)
- [14 · Run a saved .py program](#b14)
- [15 · Functions, main() and imports](#b15)
- [16 · Command arguments, paths and exit status](#b16)
- [17 · Run tests from a fresh process](#b17)
- [18 · Objects, classes, attributes and methods](#b18)


### Prepare your workspace

This setup helper is not a programming exercise. Run it once per session; in Colab, upload the complete Python Basics or weekly ZIP when prompted. Then continue with the lesson. The practice examples include sample answers; make your own prediction first.


In [ ]:
# @title Prepare the course files (run once per session)
"""Prepare the supplied course files in Jupyter or a temporary Colab runtime.

The release builder embeds this source in a notebook setup cell, then calls
prepare_course_workspace(). Importing this module alone does not upload files.
"""
from pathlib import Path, PurePosixPath
import io
import os
import shutil
import stat
import tempfile
import zipfile


_REQUIRED = (
    "programs/pickup_app.py", "programs/first_pickup.py",
    "programs/rescue_cli.py", "programs/rescue_models.py",
    "programs/test_pickup_app.py", "programs/test_rescue_models.py",
    "final-data/offer_batches.csv", "final-data/pickup_requests.csv",
    "final-data/rescue_events.csv", "assets/diagrams/python_execution_modes.svg",
)
_ALLOWED = {".py", ".csv", ".md", ".svg", ".png", ".jpg", ".jpeg", ".ipynb", ".json"}


def is_course_directory(folder):
    """Recognize the student directory without requiring a particular week."""
    folder = Path(folder)
    return all((folder / name).is_file() for name in _REQUIRED)


def extract_course_zip(data, destination_parent=None):
    """Extract a unique learning_materials/ tree into a NEW directory and return its path.

    Supports the downloadable weekly ZIP and GitHub's repository ZIP. Other
    directories are ignored. Existing work is never replaced. No archive code
    runs during extraction; running notebook cells remains a separate action.
    """
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        candidates = set()
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts
            if len(parts) >= 3 and parts[-3:] == ("learning_materials", "programs", "pickup_app.py"):
                candidates.add(parts[:-2])
        if len(candidates) != 1:
            raise ValueError("Choose a course or weekly ZIP with exactly one learning_materials/ folder. Do not upload the instructor workspace ZIP.")
        prefix = next(iter(candidates))
        files, paths, total = [], set(), 0
        for member in archive.infolist():
            raw = member.filename
            parts = PurePosixPath(raw).parts
            if parts[:len(prefix)] != prefix:
                continue
            if "\\" in raw or raw.startswith("/") or any(p in ("..", ".") or ":" in p for p in parts):
                raise ValueError("The course ZIP contains an unsafe file path.")
            relative = PurePosixPath(*parts[len(prefix):])
            if member.is_dir() or not relative.parts:
                continue
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("The course ZIP contains a symbolic link.")
            if any(p.startswith(".") or p == "__pycache__" for p in relative.parts):
                continue
            if relative.suffix.lower() not in _ALLOWED:
                continue
            if str(relative) in paths:
                raise ValueError("The course ZIP contains duplicate file paths.")
            paths.add(str(relative))
            total += member.file_size
            if total > 100 * 1024 * 1024 or len(files) >= 3000:
                raise ValueError("The course student files exceed the expected package size.")
            files.append((member, relative))
        if not all(name in paths for name in _REQUIRED):
            raise ValueError("The ZIP is missing required programs, data or diagram files. Download the complete course package.")
        session = Path(tempfile.mkdtemp(prefix="campus_food_rescue_", dir=destination_parent))
        student = session / "learning_materials"
        try:
            for member, relative in files:
                target = student.joinpath(*relative.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
        except Exception:
            shutil.rmtree(session)
            raise
    return student.resolve()


def prepare_course_workspace(force_upload=False):
    """Use a local student directory, or upload a course ZIP in Google Colab.

    force_upload=True creates a fresh Colab copy when course files are updated.
    It leaves the previous copy intact. Save your work before switching copies.
    """
    try:
        from google.colab import files
    except ImportError:
        for candidate in (Path.cwd(), Path.cwd() / "learning_materials"):
            if is_course_directory(candidate):
                os.chdir(candidate)
                print("Course files ready:", Path.cwd())
                return Path.cwd()
        raise FileNotFoundError("Open this notebook from the course learning_materials/ folder. Keep programs/, assets/ and final-data/ beside it; see SETUP.md.")
    if not force_upload and is_course_directory(Path.cwd()):
        print("Reusing this session's course files:", Path.cwd())
        return Path.cwd()
    print("Upload ONE course ZIP downloaded from GitHub or this week's course page.")
    print("Only its learning_materials/ files will be copied into a fresh temporary folder.")
    previous = Path.cwd()
    # Colab writes uploaded bytes into its current folder before returning them.
    # Keep that operation away from an existing notebook/program workspace.
    with tempfile.TemporaryDirectory(prefix="course_upload_") as upload_folder:
        try:
            os.chdir(upload_folder)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
    if len(uploaded) != 1:
        raise ValueError("Select exactly one course ZIP, then run this setup cell again.")
    name, data = next(iter(uploaded.items()))
    if not name.lower().endswith(".zip"):
        raise ValueError("Choose the .zip package, not an individual notebook or Python file.")
    folder = extract_course_zip(data, destination_parent="/content" if Path("/content").is_dir() else None)
    os.chdir(folder)
    print("Course files ready:", folder)
    print("Runtime files are temporary. Save your notebook and download any created files before leaving.")
    return folder

prepare_course_workspace()


## What to do with this reference

**Required:** use only the section named by your weekly notebook or instructor. **Already supplied:** all 19 explanations, practice examples and sample answers. **Optional:** completing the whole reference notebook; it is not a second homework stream. **Stop point:** return to the weekly task once the needed concept is clear.


<a id="b00"></a>
## 00 · Execution, comments and indentation

Run statements from top to bottom. A colon introduces an indented block.

**Watch for:** Editing a cell does not execute it.


In [ ]:
count = 4  # a comment
if count > 0:
    print(count)


[Back to topic index](#lookup-index)


<a id="b01"></a>
## 01 · Variables, types and conversion

Use = to assign; int, str and bool represent different kinds of values.

**Watch for:** "False" is nonempty text, not the Boolean False.


In [ ]:
raw = "4"
count = int(raw)
print(type(raw).__name__, type(count).__name__)
print(count * 3)


[Back to topic index](#lookup-index)


<a id="b02"></a>
## 02 · Arithmetic and precedence

Multiplication precedes addition; parentheses make grouping explicit. // and % give quotient and remainder.

**Watch for:** Quoted numbers repeat as text when multiplied by an integer.


In [ ]:
print(2 + 3 * 4, (2 + 3) * 4)
print(7 // 3, 7 % 3)
print(3 * 4, "3" * 4)


[Back to topic index](#lookup-index)


<a id="b03"></a>
## 03 · Strings and formatted output

String methods return new values. An f-string inserts values into labelled output.

**Watch for:** strip() normalizes text; it does not validate identity or meaning.


In [ ]:
raw = " bread "
name = raw.strip().upper()
count = 2
print(f"{name}: {count} containers")


[Back to topic index](#lookup-index)


<a id="b04"></a>
## 04 · Comparisons and decisions

Comparisons return Boolean values. if / elif / else chooses the first matching branch.

**Watch for:** Use == for equality; >= includes the boundary.


In [ ]:
value = 20
if value >= 20:
    label = "large"
else:
    label = "small"
print(label)


[Back to topic index](#lookup-index)


<a id="b05"></a>
## 05 · Lists, dictionaries, tuples and sets

Lists use positions; dictionaries use keys. Tuples group values; sets keep distinct values.

**Watch for:** An empty list has no item at index zero.


In [ ]:
records = [{"id": "D1", "count": 2}]
print(records[0]["count"])
print(records[0].get("window"))
identifier, count = ("D1", 2)
print(identifier, count, {"D1", "D1"})


[Back to topic index](#lookup-index)


<a id="b06"></a>
## 06 · Loops and aggregation

Initialize once before a for loop. += adds to the running total.

**Watch for:** Assigning with = inside the loop overwrites earlier contributions.


In [ ]:
total = 0
for count in [2, 0, 3]:
    total += count
print(total)


[Back to topic index](#lookup-index)


<a id="b07"></a>
## 07 · Functions, arguments and return values

def defines a callable block; arguments supply parameter values; return sends a result to the caller.

**Watch for:** Printing a result does not return that result.


In [ ]:
def product(unit, count):
    return unit * count

result = product(3, 4)
print(result)


[Back to topic index](#lookup-index)


<a id="b08"></a>
## 08 · Reading errors and debugging

Read the exception type and message, then locate the first unexpected value.

**Watch for:** Changing an expected value to match a bug hides the disagreement.


In [ ]:
expected = 5
actual = sum([2, 3])
print("Expected:", expected, "Observed:", actual)
assert actual == expected


[Back to topic index](#lookup-index)


<a id="b09"></a>
## 09 · Raising and handling exceptions

Raise a specific error for a violated contract; catch the expected exception at the appropriate boundary.

**Watch for:** Successful conversion does not establish a valid range.


In [ ]:
for raw in ["2", "0", "many"]:
    try:
        count = int(raw)
        if count <= 0:
            raise ValueError("positive count required")
        print(count)
    except ValueError as error:
        print("Rejected:", raw, str(error))


[Back to topic index](#lookup-index)


<a id="b10"></a>
## 10 · Normal, boundary, invalid and regression tests

Derive expected outcomes before executing. Check a changed case and an unaffected case.

**Watch for:** A passing normal case may miss an equality bug.


In [ ]:
def at_least_twenty(value):
    return value >= 20

for value, expected in [(19, False), (20, True), (21, True)]:
    assert at_least_twenty(value) == expected


[Back to topic index](#lookup-index)


<a id="b11"></a>
## 11 · Imports, files and CSV

Import standard-library tools. DictReader yields row dictionaries whose CSV fields are strings. Use with open(...) for a real file.

**Watch for:** CSV reading does not automatically convert numbers.


In [ ]:
import csv
from io import StringIO

text = "id,count\nD1,2\n"
for row in csv.DictReader(StringIO(text)):
    print(row["id"], int(row["count"]))


[Back to topic index](#lookup-index)


<a id="b12"></a>
## 12 · JSON, missing values and validation

json.loads converts JSON text into Python values. Check shape before fields, then type and range.

**Watch for:** JSON true becomes Python True; exclude bool when requiring an exact int.


In [ ]:
import json
candidate = json.loads('{"count": true}')
valid = isinstance(candidate, dict)
if valid:
    count = candidate.get("count")
    valid = type(count) is int and count > 0
print(valid)


[Back to topic index](#lookup-index)


<a id="b13"></a>
## 13 · SQLite, transactions and state after failure

Bind data through placeholders. with db commits on normal exit and rolls back an escaping failure. Close the connection separately.

**Watch for:** Catching a failure inside the transaction can commit partial work.


In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE stock(count INT)")
with db:
    db.execute("INSERT INTO stock VALUES (?)", (5,))
try:
    with db:
        db.execute("UPDATE stock SET count=3")
        raise RuntimeError("interruption")
except RuntimeError:
    pass
assert db.execute("SELECT count FROM stock").fetchone() == (5,)
db.close()


[Back to topic index](#lookup-index)


<a id="b14"></a>
## 14 · Run a saved .py program

Save the file, choose the course interpreter and run it from the documented directory. A script starts a fresh process.

**Watch for:** Terminal commands are not Python statements.


In [ ]:
from pathlib import Path
import sys
print("Interpreter:", sys.executable)
print("Working directory:", Path.cwd())


[Back to topic index](#lookup-index)


<a id="b15"></a>
## 15 · Functions, main() and imports

main is an ordinary function. The __name__ guard chooses direct startup; imports make definitions available.

**Watch for:** An unconditional top-level call also runs during import.


In [ ]:
def product(unit, count):
    return unit * count

def main():
    print(product(3, 4))

if __name__ == "__main__":
    main()


[Back to topic index](#lookup-index)


<a id="b16"></a>
## 16 · Command arguments, paths and exit status

argparse converts command text. Resolve relative paths from the current working directory. Check stdout, stderr and exit status.

**Watch for:** Integer parsing and domain validation are separate checks.


In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument("--count", type=int, default=4)
args = parser.parse_args(["--count", "3"])
print(args.count)


[Back to topic index](#lookup-index)


<a id="b17"></a>
## 17 · Run tests from a fresh process

Tests import the function under test. Use a test command that actually discovers your project tests.

**Watch for:** Supplied practice tests do not automatically test your own feature.


In [ ]:
import unittest
check = unittest.TestCase()
check.assertEqual(3 * 4, 12)
with check.assertRaises(ValueError):
    int("many")


[Back to topic index](#lookup-index)


<a id="b18"></a>
## 18 · Objects, classes, attributes and methods

A class describes instances. self refers to the receiving instance; a method uses its attributes.

**Watch for:** An alias shares an object; creating another instance makes a separate object.


In [ ]:
class Line:
    def __init__(self, unit, count):
        self.unit = unit
        self.count = count

    def total(self):
        return self.unit * self.count

first, second = Line(3, 4), Line(3, 2)
print(first.total(), second.total())


[Back to topic index](#lookup-index)


## Terminal patterns

Run from `learning_materials/` using your course interpreter:

```text
python programs/first_pickup.py
python programs/pickup_app.py
python programs/rescue_cli.py --containers 3
python -m unittest discover -s programs -p "test_*.py"
```

Use the [program guide](programs/README.md) for paths and the exact command for your own project. These supplied commands illustrate practice programs.

[Back to topic index](#lookup-index)
